# Reviewed companion notebook

This parallel copy retains the original workflow and pre-block SAE hooks.
It imports the `_codexgen` analyzer, whose FCA mapping closes empty extents
correctly and rejects zero-evidence support probes. Two-source meets no
longer depend on a stale third-source variable. The original notebooks and
cached data are unchanged. No model inference outputs have been regenerated.

Support floors are minima of positive sequence maxima. The `v_FG` diagnostic
summarizes matching token codes; it is not the sequence-level `F(G(p))`.
Use `tr_analyzer.fcas[layer].FG(p)` for the latter.


## Cached execution and dependencies

Use the repository uv environment (`uv sync --locked`). TransformerLens is
constrained to `>=3.9.0,<4`: version 4 removed `HookedTransformer`, still
required by this code and SAELens 6.51.1. The default execution below uses
offline caches. The six selected dense activation matrices alone occupy about
15 GB. For a bounded, one-layer execution check, use
`execute_cached_notebooks_codexgen.py` in the manuscript directory. Those
checks do not reproduce the full token-level diagnostics.


In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline


# Install libraries


```bash
conda create -n edu4 python=3.11 jupyter matplotlib
```


```bash
! pip install -U -r requirements.txt
```


```bash
! pip install -U numpy
! pip install -U scikit-learn
```


# Update repository


In [ ]:
# ! git pull


# Add import path


In [ ]:
import gc
import os
import sys
from pathlib import Path

# Read the supplied caches without making Hub requests.
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')


In [ ]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')


In [ ]:
add_library_level(level=5)


## Organize imports


In [ ]:
import multiprocessing


In [ ]:
from src.lattmc.utils import init_device, empty_cache


In [ ]:
from src.lattmc.fca.utils import *
from src.lattmc.fca.fca_utils_codexgen import *


In [ ]:
from src.lattmc.tc.transcoder_analyzers_codexgen import (
    ConceptAnalysis, init_analyzer,
)


In [ ]:
from src.lattmc.sae.nlp_sae_utils import init_device, empty_cache, gen_concept


In [ ]:
import logging


#### Number of CPU cores


In [ ]:
workers = multiprocessing.cpu_count()
workers


In [ ]:
SEED = 2026


In [ ]:
logging.basicConfig(level=logging.INFO)


In [ ]:
device = init_device()
device


In [ ]:
torch.__version__


In [ ]:
np.__version__


In [ ]:
# np.set_printoptions(precision=4, suppress=True)


# Initialize Paths


In [ ]:
OWT_PATH = Path().resolve().parents[1]
PATH = OWT_PATH / 'notebooks' / 'transcoders' / 'data'
GPT2_TR = PATH / 'transcoders' / 'gpt2'
OWT_TOKENS_DIR = GPT2_TR / 'owt_tokens'
TOKENS_PATH = OWT_TOKENS_DIR / 'owt_tokens_torch.pt'
GPT2_TR.mkdir(exist_ok=True, parents=True)
OWT_TOKENS_DIR.mkdir(exist_ok=True, parents=True)
GPT2 = Path('data') / 'sae' / 'gpt2'
GPT2.mkdir(exist_ok=True, parents=True)


In [ ]:
! ls {GPT2}


In [ ]:
GPT2.absolute()


In [ ]:
! ls {TOKENS_PATH}


In [ ]:
! ls {str(TOKENS_PATH.absolute())}


# Load trancoders


In [ ]:
layers = list(range(12))
layers = [0, 4, 6, 8, 10, 11]
# layers = list(range(12))
# layers

# Fail early instead of silently rebuilding a missing cache.
cache_files = [TOKENS_PATH]
for layer_id in layers:
    for prefix in ['V', 'v_min', 'v_max', 'v_min_nonzero']:
        cache_files.append(GPT2 / f'{prefix}{layer_id}.npz')
missing = [str(path) for path in cache_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing caches: ' + ', '.join(missing))


In [ ]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=False,
    vector_dir=GPT2,
)


In [ ]:
empty_cache()


In [ ]:
tr_analyzer.transcoder.model.cfg.device


In [ ]:
empty_cache()


# Features Dimesion


In [ ]:
tr_analyzer.fcas[0].V.shape, tr_analyzer.fcas[8].V.shape


# Analyze Tokens


In [ ]:
tr_analyzer.tokens.shape


In [ ]:
tr_analyzer.tokens[0][0]


In [ ]:
tn = to_numpy(tr_analyzer.tokens[0][0])
tn.item()


## New York City


In [ ]:
text = ' New York City'
text


In [ ]:
text_idx = 3457
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [1, 2, 3]


### New York City Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


### New York City Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    red_val=[4, 4, 4],
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(
    idcs[1:],
    vals[1:],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


### New York City Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    red_val=[4, 4, 4],
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
c = concept_an.gen_and_print(
    [2094, 10988, 10533],
    [0.38, 0.30, 0.23],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
if np.any(vals[1:2] > 0):
    concept_an.gen_and_print(
    idcs[1:2],
    vals[1:2],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=20)


## Richmond Hill


In [ ]:
text_id = 117
text = tr_analyzer.texts[text_id]
prompt = tr_analyzer.corpus[text_id]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [6, 13, 30, 31]
t_idcs = [6, 30, 31]
t_idcs = [30, 31]


### Richmond Hill Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    set_val=0.2,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


### Richmond Hill Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    set_val=4,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
gc.collect()


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(
    idcs[1:],
    vals[1:],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=100,
)


### Richmond Hill Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)


## National Park Service


In [ ]:
text_id = 21481
text = tr_analyzer.texts[text_id]
prompt = tr_analyzer.corpus[text_id]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [24, 25, 26]


### National Park Service Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    set_val=0.2,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


### National Park Service Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(
    idcs[1:],
    vals[1:],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


### National Park Service Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
concept_an.gen_and_print(
    [1944, 8298, 15520],
    [0.2, 0.2, 0.2],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
if np.any(vals[1:2] > 0):
    concept_an.gen_and_print(
    idcs[1:2],
    vals[1:2],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


## Rio de Janeiro


In [ ]:
text_idx = 5411
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 16, 17]


### Rio de Janeiro Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 0


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


#### Common Features Analysis at Layer 0


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


### Rio de Janeiro Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 8


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(
    idcs[1:],
    vals[1:],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


#### Common Features Analysis at Layer 8


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


### Rio de Janeiro Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 11


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0): 
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
if np.any(vals[1:2] > 0):
    c = concept_an.gen_and_print(
    idcs[1:2],
    vals[1:2],
    layer,
    with_text=True,
    limit=10,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)


#### Common Features Analysis at Layer 11


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


## Sport Teams Cleveland, Clippers, Cavaliers


In [ ]:
text_idx = 1924
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 39, 94]


### Sport Teams Cleveland, Clippers, Cavaliers Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 0


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


#### Common Features Analysis at Layer 0


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


### Sport Teams Cleveland, Clippers, Cavaliers Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    set_val=0.2,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 8


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
if np.any(vals[1:] > 0):
    c = concept_an.gen_and_print(
    idcs[1:],
    vals[1:],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


#### Common Features Analysis at Layer 8


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


### Sport Teams Cleveland, Clippers, Cavaliers Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 11


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
# concept_an.gen_and_print([4416, 13040, 13098], [2, 2, 2], layer,
# with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


#### Common Features Analysis at Layer 8


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


## Pets Dogs and Cats


In [ ]:
text_idx = 4042
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text


In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [ ]:
concept_an.analyze_concepts()


In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [8, 82]


### Pets Dogs and Cats Layer 0


In [ ]:
layer = 0


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 0


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_1, v_2)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


#### Common Features Analysis at Layer 0


In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


### Pets Dogs and Cats Layer 8


In [ ]:
layer = 8


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 8


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(
    [16679, 10130, 20994],
    [12, 9, 9],
    layer,
    with_text=True,
    limit=10,
)


#### Common Features Analysis at Layer 8


In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=150,
)


### Pets Dogs and Cats Layer 11


In [ ]:
layer = 11


In [ ]:
dets, vs = concept_an.gen_text(
    t_idcs,
    layer,
    limit=100,
    min_vals=True,
    full_tokens=True,
)


#### Common Features of Embeddings and Layer 11


In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer,
# with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)


In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs


In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)


#### Common Features Analysis at Layer 11


In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs


In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(
    idcs,
    vals,
    layer,
    with_text=True,
    limit=100,
)


## Full activation-vector experiments

The support-floor probes above are binary presence queries.
The following companion retains full code amplitudes and evaluates
meets, joins, four activation levels, and full-corpus statistics:

[SAE graded experiments](sae_gpt_small_tokens_places_codexgen.ipynb)

It runs independently, one layer at a time. The cell below reads
its completed manifests without loading another model or matrix.


In [ ]:
import json
from pathlib import Path

project = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'pyproject.toml').is_file()
)
result_dir = project / 'texs/sparsesurrs/lattcontexts'
result_dir /= 'experiments/activation_results'
graded_counts = []
for layer_id in [0, 8, 11]:
    path = result_dir / f'sae_layer{layer_id}.json'
    report = json.loads(path.read_text())
    graded_counts.append((layer_id, len(report['records'])))
print('(Layer, evaluated settings):', graded_counts)


## Query type and token-witness presentation

The original
`gen_text` joins reduced source probes and inspects each probe separately.
A token highlighted for one probe need not satisfy their entire join.
For a complete query `q` and token-by-coordinate matrix `trace`, use
`lattmc.contextstudy.witnesses_codexgen.classify(trace, q)` to distinguish
S (whole-query token) from yellow D (distributed item satisfaction).
Pass the original alpha when the experiment scales the query. This
companion does not replace the original support-floor queries with
full-amplitude queries. The recorded full-code examples and independently
replayed Cat/dog cases have separate provenance in the paper.